# Notebook 05: anomalies in time, one-liners against a temporal autoencoder

**Class 03_02.** A value can be normal in itself and anomalous *at that moment*: 80 requests per second at noon is
normal, at 3 a.m. it is not (a **contextual** anomaly). Temporal detectors look at a **window** of the series.

**Data:** four artificial series of the Numenta Anomaly Benchmark (NAB, Lavin and Ahmad 2015), 5-minute samples over
14 days, a daily pattern (high by day, low at night), and one anomaly each:
`jumpsup` (a day goes too high), `jumpsdown` (too low), `nojump` (the daily rise does not happen), `flatmiddle` (the
signal stays flat). NAB labels a tolerance **window** around each anomaly.

**Protocol (as NAB):** the first 8 days are clean training data, day 9 sets the threshold (its maximum score + 10%),
days 1-14 are scored. We report whether the anomaly is **found** (an alarm inside the window), the **delay** from
the window start, and the **false alarms** (alarms outside the window).

In [ ]:
import os
import sys

os.environ.setdefault("KERAS_BACKEND", "jax")
import numpy as np
import pandas as pd
import polars as pl

sys.path[:0] = [".", "../../src"]  # this folder and the generic libraries of the repository
import aasplot as ap
import anomlib as an
import deeplib as dl
from aasplot import BLUE, GREY, LIGHT, RED

an.table_config()
ap.setup("03")
D, W = dl.DAY, dl.DAY  # one day = 288 samples = the window length
SERIES = ["jumpsup", "jumpsdown", "nojump", "flatmiddle"]

In [ ]:
def nab_series(series: dict) -> None:
    """series: {name: DataFrame(timestamp, value, y)}; the NAB label window is shaded."""
    f, axs = ap.fig("full", 1.25, nrows=2, ncols=2, sharex=True, gridspec_kw={"hspace": 0.4, "wspace": 0.25})
    for ax, (name, te) in zip(axs.ravel(), series.items()):
        ax.plot(te.timestamp, te.value, lw=0.6, color=BLUE)
        ax.fill_between(te.timestamp, te.value.min(), te.value.max(), where=te.y == 1, color=RED, alpha=0.25, lw=0)
        ax.set_title(name, fontsize=7.5, loc="left", pad=1)
        ax.set(yticks=[], xticks=[])
        ax.grid(False)
    ap.save(f, "nab_series")


nab_series({n: dl.load_nab(n)[1] for n in SERIES})

## A. Three one-liners

* $|x_t|$ (z-score of the value): catches values outside the normal range only.
* **seasonal difference** $|x_t-x_{t-288}|$: compares with the same time yesterday.
* **moving standard deviation** over 1 hour: catches changes of variability.

## B. A convolutional autoencoder over one-day windows

Input: a window of 288 samples. Encoder: two 1-D convolutions with stride 2 (288 $\to$ 144 $\to$ 72 steps),
decoder: two transposed convolutions back to 288. The score of time step $t$ is the squared reconstruction error
**at $t$**, averaged over the 288 windows that contain it (averaging the *window* error instead smears an anomaly
over two days and causes hundreds of false alarms).

In [ ]:
ONE = {"|x|": lambda x: np.abs(x),
       "seasonal diff": lambda x: np.r_[np.full(D, np.nan), np.abs(x[D:] - x[:-D])],
       "moving std 1h": lambda x: pd.Series(x).rolling(12, min_periods=2).std().to_numpy()}  # fmt: skip
rows, curves = [], {}
for name in SERIES:
    _, te = dl.load_nab(name)
    v, y = te.value.to_numpy(), te.y.to_numpy()
    mu, sd = v[: 8 * D].mean(), v[: 8 * D].std()
    x = (v - mu) / sd
    m = dl.fit_window_ae(dl.conv_ae(W, 1), dl.windows(x[: 8 * D], W), epochs=30, seed=0)
    fs = dict(ONE)
    fs["conv AE"] = lambda x, m=m: dl.window_to_point(dl.window_error(m, dl.windows(x, W)), W, len(x))
    for k, f in fs.items():
        s = f(x)
        t = np.nanmax(s[8 * D : 9 * D]) * 1.1
        rows.append({"series": name, "detector": k, **dl.series_report(np.nan_to_num(s), y, t)})
        curves[(name, k)] = (s, t)
res = pl.DataFrame(rows)
print(res.pivot(on="detector", index="series", values="found"))
print(res.pivot(on="detector", index="series", values="false_alarms"))
print(res.pivot(on="detector", index="series", values="delay"))

In [ ]:
import seaborn as sns
from matplotlib.patches import Rectangle


def nab_found(rows) -> None:
    """rows: dicts with series, detector, found (0/1), false_alarms: false alarms per cell, `missed` where nothing was found."""
    dets = list(dict.fromkeys(r["detector"] for r in rows))
    sers = list(dict.fromkeys(r["series"] for r in rows))
    fa = np.array(
        [[next(r["false_alarms"] for r in rows if r["series"] == s and r["detector"] == d) for d in dets] for s in sers]
    )
    fnd = np.array(
        [[next(r["found"] for r in rows if r["series"] == s and r["detector"] == d) for d in dets] for s in sers]
    )
    f, ax = ap.fig("full", 1.35)
    sns.heatmap(
        np.log1p(fa),
        mask=fnd == 0,
        cmap=sns.light_palette(RED, as_cmap=True),
        cbar=False,
        linewidths=1,
        linecolor="white",
        ax=ax,
        vmin=0,
        vmax=np.log1p(400),
    )
    for i in range(len(sers)):
        for j in range(len(dets)):
            ax.text(
                j + 0.5,
                i + 0.5,
                "missed" if fnd[i, j] == 0 else f"found, {int(fa[i, j])}",
                ha="center",
                va="center",
                fontsize=7,
                color=GREY if fnd[i, j] == 0 else "black",
            )
            if fnd[i, j] == 0:
                ax.add_patch(Rectangle((j, i), 1, 1, fc=LIGHT, alpha=0.35, lw=0))
    ax.set_xticks(np.arange(len(dets)) + 0.5, dets)
    ax.set_yticks(np.arange(len(sers)) + 0.5, sers, rotation=0)
    ax.xaxis.tick_top()
    ax.tick_params(length=0)
    ax.grid(False)
    ap.save(f, "nab_found")


nab_found(res.to_dicts())

In [ ]:
(s_d, t_d), (s_a, t_a) = curves[("nojump", "seasonal diff")], curves[("nojump", "conv AE")]

In [ ]:
def nab_scores(s_diff, s_ae, t_diff, t_ae, y) -> None:
    """The score of the seasonal difference and of the conv AE on one series, with threshold and anomaly window."""
    f, axs = ap.fig("full", 1.1, nrows=2, sharex=True, gridspec_kw={"hspace": 0.5})
    win = np.flatnonzero(y)[[0, -1]]
    for ax, s, t, name in ((axs[0], s_diff, t_diff, "seasonal difference"), (axs[1], s_ae, t_ae, "conv AE")):
        ax.plot(s, lw=0.6, color=BLUE)
        ax.axhline(t, color=GREY, ls="--", lw=0.9)
        ax.axvspan(*win, color=RED, alpha=0.25, lw=0)
        ax.set_title(name, fontsize=7.5, loc="left", pad=1)
        ax.set(yticks=[])
        ax.grid(False)
    axs[1].set_xlabel("time step (5 min); red: the anomaly window, dashed: threshold")
    ap.save(f, "nab_scores")


nab_scores(s_d, s_a, t_d, t_a, dl.load_nab("nojump")[1].y.to_numpy())

**Reading.**
* $|x|$ and the moving standard deviation find only the jump up: the other anomalies stay inside the normal range.
* The **seasonal difference finds all four**, with one weakness: the day *after* the anomaly is compared with the
  anomalous day, so it raises an "echo" of about 110 false alarms.
* The **conv AE finds all four with 1 to 35 false alarms**: it learned the shape of a normal day, so a missing rise
  or a flat stretch cannot be reconstructed.
* A one-liner that encodes the right prior (daily seasonality) gets most of the way. Benchmarks of time-series
  anomaly detection find the same: simple methods are often as good as deep ones (Wu and Keogh 2021; TSB-AD,
  Liu and Paparrizos 2024), and time-series *foundation models* used zero-shot do not beat moving-window one-liners
  (ICLR 2026). Always run the one-liners first.

## C. A recurrent autoencoder (LSTM), for comparison

In [ ]:
_, te = dl.load_nab("nojump")
v, y = te.value.to_numpy(), te.y.to_numpy()
x = (v - v[: 8 * D].mean()) / v[: 8 * D].std()
W2 = 96  # 8 hours: an LSTM over 288 steps is slow on a CPU
lstm = dl.fit_window_ae(dl.lstm_ae(W2, 1, units=32), dl.windows(x[: 8 * D], W2)[::4], epochs=20, seed=0)
s = dl.window_to_point(dl.window_error(lstm, dl.windows(x, W2)), W2, len(x))
print("LSTM AE on nojump:", dl.series_report(s, y, s[8 * D : 9 * D].max() * 1.1))

The LSTM AE (8-hour windows, to keep it fast on a CPU) does **not** find the missing rise: an 8-hour window does not
hold a whole day, so "no rise today" looks like a quiet morning. The window must cover the context that defines
normal; the architecture matters less.

## D. Deeper and residual bodies, and an ISODATA cut on a series

`build_autoencoder` builds the temporal bodies too: `kind="conv"` (strided 1-D convolutions, one stage halves the
window), `"lstm"` and `"gru"`, each with `residual=True` (skip connections) and `norm="layer"`. Below, three of
them on all four series, with two thresholds: the NAB protocol (maximum score of the clean day 9 plus 10%) and
**ISODATA on $\log$ score** of the whole series, which needs no clean day but assumes the anomaly is a visible
second mode (here about 10% of the steps).

In [ ]:
import time

import thresholds as th
from autoenc import build_autoencoder

ARCH = {"conv, 2 stages": dict(kind="conv", widths=(32, 16), code=None, activation="relu", w=288),
        "conv, 2 stages, SiLU": dict(kind="conv", widths=(32, 16), code=None, w=288),
        "residual conv, 3 stages": dict(kind="conv", widths=(32, 16, 8), code=None, activation="relu", residual=True, norm="layer", w=288),
        "GRU, 8 h window": dict(kind="gru", widths=(32,), code=16, w=96)}  # fmt: skip
rows = []
for name in SERIES:
    _, te = dl.load_nab(name)
    v, y = te.value.to_numpy(), te.y.to_numpy()
    x = (v - v[: 8 * D].mean()) / v[: 8 * D].std()
    for arch, cfg in ARCH.items():
        cfg = dict(cfg)
        w = cfg.pop("w")
        t0 = time.time()
        m = dl.fit_window_ae(build_autoencoder((w, 1), **cfg), dl.windows(x[: 8 * D], w)[:: 4 if cfg["kind"] == "gru" else 1],
                             epochs=30, seed=0)  # fmt: skip
        s = dl.window_to_point(dl.window_error(m, dl.windows(x, w)), w, len(x))
        for rule, t in (
            ("day 9 max + 10%", s[8 * D : 9 * D].max() * 1.1),
            ("ISODATA, log score", th.isodata(s[w:], log=True)),
        ):
            rows.append(
                {
                    "series": name,
                    "body": arch,
                    "threshold": rule,
                    **dl.series_report(s, y, t),
                    "seconds": time.time() - t0,
                }
            )
arch = pl.DataFrame(rows)
print(arch.group_by(["body", "threshold"], maintain_order=True)
      .agg(pl.col("found").sum().alias("found of 4"), pl.col("delay").mean().round(0), pl.col("false_alarms").mean().round(0)))  # fmt: skip

**Reading.** With the NAB threshold only the plain ReLU conv body finds all four series (10 false alarms in total).
The same body with SiLU finds two, the 3-stage residual conv one and the GRU one: a smoother or deeper network
*generalises* so well that it rebuilds the anomaly too, and the first four series are the kind where that
hurts. Architecture is a hyper-parameter to test against the one-liners, not an upgrade. ISODATA on the log score
finds all four but raises 990 to 2,700 false alarms: a series is not bimodal in score (the anomaly is a smooth
stretch among values that vary continuously), so the cut falls inside the normal range. On a series, take the
threshold from a clean day.